# Fabric notebook: score net revenue and write predictions to the Lakehouse

Run this **inside a Microsoft Fabric notebook** attached to a Lakehouse. It
reads snapshot inputs from `Files/revenue/input`, scores them, and writes a
Power BI-ready predictions table to `Files/revenue/predictions`.

**All data is synthetic.** Replace placeholders with your own paths. Do not
commit real workspace/lakehouse identifiers.

> In Fabric, the attached Lakehouse is available under `/lakehouse/default/`.
> Install the accelerator into the Fabric environment (e.g. `%pip install`) or
> attach it as a custom library.

In [ ]:
# %pip install revenue-prediction-accelerator  # or attach as an environment library
import pandas as pd

INPUT_PATH = '/lakehouse/default/Files/revenue/input/revenue_snapshots.parquet'
OUTPUT_PATH = '/lakehouse/default/Files/revenue/predictions/predictions.parquet'

snapshots = pd.read_parquet(INPUT_PATH)
snapshots.head()

In [ ]:
# Load the registered/exported model bundle. In production, load the MLflow
# model registered in Azure ML; here we load a champion bundle staged in the
# Lakehouse for demonstration.
from revenue_prediction.inference.predict import load_bundle, batch_predict

bundle = load_bundle('/lakehouse/default/Files/revenue/models/champion_bundle.joblib')
predictions = batch_predict(bundle, snapshots, cutoff_day=15)
predictions.head()

In [ ]:
predictions.to_parquet(OUTPUT_PATH, index=False)
print('Wrote', len(predictions), 'predictions to', OUTPUT_PATH)

Build a **DirectLake** semantic model over the predictions table, then a Power
BI report. All outputs are synthetic and for demonstration only.